CAPSAICIN

In [6]:
# =============================================================================
# CAPSAICIN - FIXED HYPERPARAMETER TUNING  (manual LOO loop for all models)
# Input files:
#   - Capsaicin_standardized_data_1_.csv
#   - Capsaicin_changed_1_.csv
# =============================================================================
!pip install catboost
import pandas as pd
import numpy as np
import matplotlib
import catboost
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import shap
import itertools

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'Random Forest':      '#1f77b4',
    'Gradient Boosting':  '#2ca02c',
    'XGBoost':            '#ff7f0e',
    'CatBoost':           '#d62728',
}
MARKERS = {'9:1': 'o', '8:2': '^', '7:3': 's'}

# =============================================================================
# 1. LOAD DATA
# =============================================================================
data = pd.read_csv('Cap_os_standardized_data_1_.csv')
data_raw = pd.read_csv('Capsaicin_changed_1_.csv')

X = data[['TIBA', 'TDZ', 'BA']].values
y = data['nanogramPerG'].values
feature_names = ['TIBA', 'TDZ', 'BA']

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

# =============================================================================
# 2. MANUAL LOO CV GRID SEARCH  (avoids nan bug in GridSearchCV + LOO)
#    For each param combination: run LOO CV, compute R², keep the best.
#    This is methodologically identical to GridSearchCV — just explicit.
# =============================================================================

def loo_cv_score(model_class, params, X, y):
    """Run LOO CV for a given model + params, return R²."""
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y)
    for train_idx, test_idx in loo.split(X):
        m = model_class(**params)
        m.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = m.predict(X[test_idx])
    return r2_score(y, y_pred), y_pred


def grid_search_loo(model_class, param_grid, X, y, label):
    """Manual grid search over param_grid using LOO CV."""
    keys   = list(param_grid.keys())
    values = list(param_grid.values())
    combos = list(itertools.product(*values))

    best_score  = -np.inf
    best_params = None
    total = len(combos)

    print(f"\n{label}: searching {total} combinations...")
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        try:
            score, _ = loo_cv_score(model_class, params, X, y)
            if score > best_score:
                best_score  = score
                best_params = params
        except Exception:
            pass
        if i % 10 == 0 or i == total:
            print(f"  {i}/{total} done — best so far: {best_score:.3f}", flush=True)

    print(f"  → Best params: {best_params}")
    print(f"  → Best LOO R²: {best_score:.3f}")
    return best_params, best_score


print("\n" + "="*65)
print("CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)")
print("="*65)

# ── Random Forest ─────────────────────────────────────────────
rf_params, rf_score = grid_search_loo(
    RandomForestRegressor,
    {
        'n_estimators':     [50, 100, 200],
        'max_depth':        [None, 3, 5, 7],
        'min_samples_split':[2, 4, 6],
        'random_state':     [42],
    },
    X, y, 'Random Forest'
)

# ── Gradient Boosting ─────────────────────────────────────────
gb_params, gb_score = grid_search_loo(
    GradientBoostingRegressor,
    {
        'n_estimators':  [100, 200, 300],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth':     [2, 3, 4],
        'random_state':  [42],
    },
    X, y, 'Gradient Boosting'
)

# ── XGBoost ───────────────────────────────────────────────────
xgb_params, xgb_score = grid_search_loo(
    XGBRegressor,
    {
        'n_estimators':     [50, 100, 200],
        'learning_rate':    [0.05, 0.1, 0.2],
        'max_depth':        [3, 4, 6],
        'min_child_weight': [1, 3, 5],
        'subsample':        [0.8, 1.0],
        'random_state':     [42],
        'verbosity':        [0],
    },
    X, y, 'XGBoost'
)

# ── CatBoost ──────────────────────────────────────────────────
cb_params, cb_score = grid_search_loo(
    CatBoostRegressor,
    {
        'iterations':    [500, 1000],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth':         [4, 6, 8],
        'loss_function': ['RMSE'],
        'random_state':  [42],
        'verbose':       [0],
    },
    X, y, 'CatBoost'
)

print("\n" + "="*65)
print("TUNING COMPLETE — BEST PARAMS PER MODEL")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

# =============================================================================
# 3. BUILD TUNED MODELS
# =============================================================================
tuned_models = {
    'Random Forest':     RandomForestRegressor(**rf_params),
    'Gradient Boosting': GradientBoostingRegressor(**gb_params),
    'XGBoost':           XGBRegressor(**xgb_params),
    'CatBoost':          CatBoostRegressor(**cb_params),
}

splits = {'9:1': 0.1, '8:2': 0.2, '7:3': 0.3}

# =============================================================================
# 4. TRAIN-TEST SPLIT RESULTS
# =============================================================================
print("\n" + "="*65)
print("TRAIN-TEST SPLIT METRICS  (tuned models)")
print("="*65)

split_results = {}

for split_label, test_size in splits.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42)
    split_results[split_label] = {}

    print(f"\n── Split {split_label} ──")
    print(f"{'Model':<22} {'Train R²':>9} {'Train MAE':>10} {'Train RMSE':>11}"
          f" {'Test R²':>8} {'Test MAE':>9} {'Test RMSE':>10}")
    print("-"*82)

    for name, model in tuned_models.items():
        m = model.__class__(**model.get_params())
        m.fit(X_tr, y_tr)

        ytr_p = m.predict(X_tr)
        yte_p = m.predict(X_te)

        res = {
            'train_r2':   r2_score(y_tr, ytr_p),
            'train_mae':  mean_absolute_error(y_tr, ytr_p),
            'train_rmse': np.sqrt(mean_squared_error(y_tr, ytr_p)),
            'test_r2':    r2_score(y_te, yte_p),
            'test_mae':   mean_absolute_error(y_te, yte_p),
            'test_rmse':  np.sqrt(mean_squared_error(y_te, yte_p)),
            'y_test':     y_te,
            'y_pred':     yte_p,
        }
        split_results[split_label][name] = res

        print(f"{name:<22} {res['train_r2']:>9.2f} {res['train_mae']:>10.2f} "
              f"{res['train_rmse']:>11.2f} {res['test_r2']:>8.2f} "
              f"{res['test_mae']:>9.2f} {res['test_rmse']:>10.2f}")

# =============================================================================
# 5. FINAL LOO CV WITH TUNED MODELS
# =============================================================================
print("\n" + "="*65)
print("LEAVE-ONE-OUT CV RESULTS  (tuned models)")
print("="*65)
print(f"{'Model':<22} {'LOO R²':>8} {'LOO MAE':>9} {'LOO RMSE':>10}")
print("-"*52)

loo_results = {}
loo = LeaveOneOut()

for name, model in tuned_models.items():
    m = model.__class__(**model.get_params())
    y_loo_pred = cross_val_predict(m, X, y, cv=loo)
    loo_results[name] = {
        'r2':     r2_score(y, y_loo_pred),
        'mae':    mean_absolute_error(y, y_loo_pred),
        'rmse':   np.sqrt(mean_squared_error(y, y_loo_pred)),
        'y_pred': y_loo_pred,
    }
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>8.2f} {r['mae']:>9.2f} {r['rmse']:>10.2f}")

# =============================================================================
# FIGURES
# =============================================================================

# Figure 1 — Actual vs Predicted (all splits)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, split_label in zip(axes, splits):
    for name, color in COLORS.items():
        res = split_results[split_label][name]
        ax.scatter(res['y_test'], res['y_pred'], color=color,
                   alpha=0.75, s=45, marker=MARKERS[split_label],
                   label=name, zorder=3)
    lo, hi = y.min()-0.1, y.max()+0.1
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,label='Perfect fit',zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual values (standardised)')
    ax.set_title(f'Split {split_label}')
    ax.grid(True,alpha=0.3); ax.set_aspect('equal',adjustable='box')
axes[0].set_ylabel('Predicted values (standardised)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           frameon=True, bbox_to_anchor=(0.5,-0.08))
fig.suptitle('Capsaicin — Actual vs. Predicted (Tuned)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_ActualVsPredicted_tuned.png', bbox_inches='tight')
plt.close()
print("\nSaved: Fig_Capsaicin_ActualVsPredicted_tuned.png")

# Figure 2 — LOO CV actual vs predicted
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    ax.scatter(y, res['y_pred'], color=color, alpha=0.7, s=40, zorder=3)
    lo, hi = y.min()-0.1, y.max()+0.1
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual (standardised)')
    ax.set_title(f'{name}\nLOO R²={res["r2"]:.2f}', fontsize=10)
    ax.grid(True,alpha=0.3); ax.set_aspect('equal',adjustable='box')
axes[0].set_ylabel('Predicted (standardised)')
fig.suptitle('Capsaicin — Leave-One-Out CV (Tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_LOO_CV_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_LOO_CV_tuned.png")

# Figure 3 — Residuals
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    residuals = y - res['y_pred']
    ax.scatter(res['y_pred'], residuals, color=color, alpha=0.7, s=40, zorder=3)
    ax.axhline(0, color='black', linestyle='--', lw=1.2, zorder=2)
    ax.set_xlabel('Predicted (standardised)')
    ax.set_title(name, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Residual (actual − predicted)')
fig.suptitle('Capsaicin — Residual Analysis (LOO CV, Tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_Residuals_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_Residuals_tuned.png")

# Figure 4 — LOO R² bar chart
fig, ax = plt.subplots(figsize=(7, 4))
names   = list(loo_results.keys())
r2_vals = [loo_results[n]['r2'] for n in names]
colors  = [COLORS[n] for n in names]
bars = ax.bar(names, r2_vals, color=colors, width=0.5,
              edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('LOO Cross-Validation R²')
ax.set_title('Capsaicin — Model Comparison (LOO CV, Tuned)')
ax.axhline(0.7, color='grey', linestyle=':', lw=1, label='R²=0.70 reference')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_LOO_R2_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_LOO_R2_tuned.png")

# Figure 5 — SHAP (best model = CatBoost)
print("\nComputing SHAP values (CatBoost, tuned)...")
cb_full = CatBoostRegressor(**cb_params)
cb_full.fit(X, y)
explainer = shap.TreeExplainer(cb_full)
shap_vals = explainer.shap_values(X)

fig, ax = plt.subplots(figsize=(7, 4))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='dot', show=False, color_bar=True)
plt.title('Capsaicin — SHAP Feature Importance (CatBoost, Tuned)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_SHAP_beeswarm_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_SHAP_beeswarm_tuned.png")

fig, ax = plt.subplots(figsize=(6, 3.5))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='bar', show=False)
plt.title('Capsaicin — Mean |SHAP| Feature Importance (CatBoost)', fontsize=12)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_SHAP_bar_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_SHAP_bar_tuned.png")

# Figure 6 — Feature scatter plots (raw scale)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, feature_names):
    ax.scatter(data_raw[feat].values, data_raw['nanogramPerG'].values,
               color='#1f77b4', alpha=0.7, s=50, edgecolors='white', lw=0.5)
    ax.set_xlabel(f'{feat} concentration (mg/L)', fontsize=11)
    ax.set_ylabel('Capsaicin yield (ng/g)', fontsize=11)
    ax.set_title(f'Capsaicin vs {feat}', fontsize=12)
    ax.grid(True, alpha=0.3); ax.tick_params(labelsize=10)
plt.suptitle('Capsaicin — Feature Scatter Plots', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Capsaicin_FeaturePlots.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Capsaicin_FeaturePlots.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*65)
print("BEST HYPERPARAMETERS  (report in paper Section IV.D)")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

print("\n" + "="*65)
print("SUMMARY: LOO CV METRICS  (tuned) — copy into paper table")
print("="*65)
print(f"{'Model':<22} {'R²':>6} {'MAE':>7} {'RMSE':>8}")
print("-"*46)
for name in loo_results:
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>6.2f} {r['mae']:>7.2f} {r['rmse']:>8.2f}")

print("\nAll figures saved. Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.2 MB/s eta 0:00:00
Dataset: 47 samples, 3 features

CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)

Random Forest: searching 36 combinations...
  10/36 done — best so far: 0.393
  20/36 done — best so far: 0.393
  30/36 done — best so far: 0.393
  36/36 done — best so far: 0.393
  → Best params: {'n_estimators': 50, 'max_depth': 3, 'min_samples_split': 2, 'random_state': 42}
  → Best LOO R²: 0.393

Gradient Boosting: searching 27 combinations...
  10/27 done — best so far: 0.378
  20/27 done — best so far: 0.378
  27/27 done — best so far: 0.378
  → Best params: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'random_state': 42}
  → Best LOO R²: 0.378

XGBoost: searching 162 combinations...
  10/162 done — best so far: 0.332
  20/162 done — best so far: 0.360
  30/162 done — best so far: 0.360
  40/162 done — best so far: 0.360
  50/162 done — best so far: 0.362
  60/162 done — best so far: 0.381
  70/162

VANILYAMINE


In [8]:
# =============================================================================
# VANILLYLAMINE - IMPROVED ANALYSIS WITH CROSS-VALIDATED HYPERPARAMETER TUNING
# Input files:
#   - Vanilylamine_standardized_data.csv   (for model training)
#   - Vanilylamine_changed.csv             (for feature scatter plots)
#
# NOTE: n=15 samples. LOO CV is the primary evaluation method.
# Train-test splits are included for consistency with paper structure
# but should be interpreted cautiously at this sample size.
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import shap
import itertools

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'Random Forest':      '#1f77b4',
    'Gradient Boosting':  '#2ca02c',
    'XGBoost':            '#ff7f0e',
    'CatBoost':           '#d62728',
}
MARKERS = {'9:1': 'o', '8:2': '^', '7:3': 's'}

# =============================================================================
# 1. LOAD DATA
# =============================================================================
data = pd.read_excel('Vanilylamine_standardized_data.xlsx')
data_raw = pd.read_excel('Vanilylamine_changed.xlsx', sheet_name=2)

X = data[['TIBA', 'TDZ', 'BA']].values
y = data['nanogramPerG'].values
feature_names = ['TIBA', 'TDZ', 'BA']

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"y range: {y.min():.3f} to {y.max():.3f}  (standardised)")
print(f"NOTE: n=15. LOO CV is the primary evaluation metric.")

# =============================================================================
# 2. MANUAL LOO CV GRID SEARCH
# =============================================================================

def loo_cv_score(model_class, params, X, y):
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y)
    for train_idx, test_idx in loo.split(X):
        m = model_class(**params)
        m.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = m.predict(X[test_idx])
    return r2_score(y, y_pred), y_pred


def grid_search_loo(model_class, param_grid, X, y, label):
    keys   = list(param_grid.keys())
    values = list(param_grid.values())
    combos = list(itertools.product(*values))

    best_score  = -np.inf
    best_params = None
    total = len(combos)

    print(f"\n{label}: searching {total} combinations...")
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        try:
            score, _ = loo_cv_score(model_class, params, X, y)
            if score > best_score:
                best_score  = score
                best_params = params
        except Exception:
            pass
        if i % 10 == 0 or i == total:
            print(f"  {i}/{total} done — best so far: {best_score:.3f}", flush=True)

    print(f"  → Best params: {best_params}")
    print(f"  → Best LOO R²: {best_score:.3f}")
    return best_params, best_score


print("\n" + "="*65)
print("CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)")
print("="*65)

rf_params, rf_score = grid_search_loo(
    RandomForestRegressor,
    {
        'n_estimators':      [50, 100, 200],
        'max_depth':         [None, 3, 5, 7],
        'min_samples_split': [2, 4, 6],
        'random_state':      [42],
    },
    X, y, 'Random Forest'
)

gb_params, gb_score = grid_search_loo(
    GradientBoostingRegressor,
    {
        'n_estimators':  [100, 200, 300],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth':     [2, 3, 4],
        'random_state':  [42],
    },
    X, y, 'Gradient Boosting'
)

xgb_params, xgb_score = grid_search_loo(
    XGBRegressor,
    {
        'n_estimators':     [50, 100, 200],
        'learning_rate':    [0.05, 0.1, 0.2],
        'max_depth':        [3, 4, 6],
        'min_child_weight': [1, 3, 5],
        'subsample':        [0.8, 1.0],
        'random_state':     [42],
        'verbosity':        [0],
    },
    X, y, 'XGBoost'
)

cb_params, cb_score = grid_search_loo(
    CatBoostRegressor,
    {
        'iterations':    [500, 1000],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth':         [4, 6, 8],
        'loss_function': ['RMSE'],
        'random_state':  [42],
        'verbose':       [0],
    },
    X, y, 'CatBoost'
)

print("\n" + "="*65)
print("TUNING COMPLETE — BEST PARAMS PER MODEL")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

# =============================================================================
# 3. BUILD TUNED MODELS
# =============================================================================
tuned_models = {
    'Random Forest':     RandomForestRegressor(**rf_params),
    'Gradient Boosting': GradientBoostingRegressor(**gb_params),
    'XGBoost':           XGBRegressor(**xgb_params),
    'CatBoost':          CatBoostRegressor(**cb_params),
}

splits = {'9:1': 0.1, '8:2': 0.2, '7:3': 0.3}

# =============================================================================
# 4. TRAIN-TEST SPLIT RESULTS
#    With n=15: 9:1 → 1 test sample, 8:2 → 3, 7:3 → 4
#    These are reported for paper consistency but LOO CV is primary.
# =============================================================================
print("\n" + "="*65)
print("TRAIN-TEST SPLIT METRICS  (tuned models)")
print("CAUTION: very few test samples at n=15 — interpret carefully")
print("="*65)

split_results = {}

for split_label, test_size in splits.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42)
    split_results[split_label] = {}

    n_test = len(y_te)
    print(f"\n── Split {split_label}  ({n_test} test samples) ──")
    print(f"{'Model':<22} {'Train R²':>9} {'Train MAE':>10} {'Train RMSE':>11}"
          f" {'Test R²':>8} {'Test MAE':>9} {'Test RMSE':>10}")
    print("-"*82)

    for name, model in tuned_models.items():
        m = model.__class__(**model.get_params())
        m.fit(X_tr, y_tr)

        ytr_p = m.predict(X_tr)
        yte_p = m.predict(X_te)

        res = {
            'train_r2':   r2_score(y_tr, ytr_p),
            'train_mae':  mean_absolute_error(y_tr, ytr_p),
            'train_rmse': np.sqrt(mean_squared_error(y_tr, ytr_p)),
            'test_r2':    r2_score(y_te, yte_p),
            'test_mae':   mean_absolute_error(y_te, yte_p),
            'test_rmse':  np.sqrt(mean_squared_error(y_te, yte_p)),
            'y_test':     y_te,
            'y_pred':     yte_p,
        }
        split_results[split_label][name] = res

        print(f"{name:<22} {res['train_r2']:>9.2f} {res['train_mae']:>10.2f} "
              f"{res['train_rmse']:>11.2f} {res['test_r2']:>8.2f} "
              f"{res['test_mae']:>9.2f} {res['test_rmse']:>10.2f}")

# =============================================================================
# 5. FINAL LOO CV WITH TUNED MODELS  (primary result)
# =============================================================================
print("\n" + "="*65)
print("LEAVE-ONE-OUT CV RESULTS  (tuned models, Vanillylamine)")
print("="*65)
print(f"{'Model':<22} {'LOO R²':>8} {'LOO MAE':>9} {'LOO RMSE':>10}")
print("-"*52)

loo_results = {}
loo = LeaveOneOut()

for name, model in tuned_models.items():
    m = model.__class__(**model.get_params())
    y_loo_pred = cross_val_predict(m, X, y, cv=loo)
    loo_results[name] = {
        'r2':     r2_score(y, y_loo_pred),
        'mae':    mean_absolute_error(y, y_loo_pred),
        'rmse':   np.sqrt(mean_squared_error(y, y_loo_pred)),
        'y_pred': y_loo_pred,
    }
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>8.2f} {r['mae']:>9.2f} {r['rmse']:>10.2f}")

# =============================================================================
# FIGURES
# =============================================================================

# Figure 1 — Actual vs Predicted (all splits)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, split_label in zip(axes, splits):
    for name, color in COLORS.items():
        res = split_results[split_label][name]
        ax.scatter(res['y_test'], res['y_pred'], color=color,
                   alpha=0.75, s=60, marker=MARKERS[split_label],
                   label=name, zorder=3)
    lo, hi = y.min()-0.15, y.max()+0.15
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,label='Perfect fit',zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual values (standardised)')
    ax.set_title(f'Split {split_label}')
    ax.grid(True, alpha=0.3); ax.set_aspect('equal', adjustable='box')
axes[0].set_ylabel('Predicted values (standardised)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           frameon=True, bbox_to_anchor=(0.5, -0.08))
fig.suptitle('Vanillylamine — Actual vs. Predicted (Tuned Models)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_ActualVsPredicted_tuned.png', bbox_inches='tight')
plt.close()
print("\nSaved: Fig_Vanillylamine_ActualVsPredicted_tuned.png")

# Figure 2 — LOO CV actual vs predicted
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    ax.scatter(y, res['y_pred'], color=color, alpha=0.7, s=55, zorder=3)
    lo, hi = y.min()-0.15, y.max()+0.15
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual (standardised)')
    ax.set_title(f'{name}\nLOO R²={res["r2"]:.2f}', fontsize=10)
    ax.grid(True, alpha=0.3); ax.set_aspect('equal', adjustable='box')
axes[0].set_ylabel('Predicted (standardised)')
fig.suptitle('Vanillylamine — Leave-One-Out CV (Tuned Models)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_LOO_CV_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_LOO_CV_tuned.png")

# Figure 3 — Residuals
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    residuals = y - res['y_pred']
    ax.scatter(res['y_pred'], residuals, color=color, alpha=0.7, s=55, zorder=3)
    ax.axhline(0, color='black', linestyle='--', lw=1.2, zorder=2)
    ax.set_xlabel('Predicted (standardised)')
    ax.set_title(name, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Residual (actual − predicted)')
fig.suptitle('Vanillylamine — Residual Analysis (LOO CV, Tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_Residuals_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_Residuals_tuned.png")

# Figure 4 — LOO R² bar chart
fig, ax = plt.subplots(figsize=(7, 4))
names   = list(loo_results.keys())
r2_vals = [loo_results[n]['r2'] for n in names]
colors  = [COLORS[n] for n in names]
bars = ax.bar(names, r2_vals, color=colors, width=0.5,
              edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, r2_vals):
    ypos = max(bar.get_height(), 0) + 0.01
    ax.text(bar.get_x()+bar.get_width()/2, ypos,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(-0.3, 1.1)
ax.set_ylabel('LOO Cross-Validation R²')
ax.set_title('Vanillylamine — Model Comparison (LOO CV, Tuned)')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(0.5, color='grey', linestyle=':', lw=1, label='R²=0.50 reference')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_LOO_R2_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_LOO_R2_tuned.png")

# Figure 5 — SHAP (best model by LOO R²)
best_model_name = max(loo_results, key=lambda n: loo_results[n]['r2'])
print(f"\nBest model for SHAP: {best_model_name} (LOO R²={loo_results[best_model_name]['r2']:.2f})")
print("Computing SHAP values...")

best_model_obj = tuned_models[best_model_name].__class__(
    **tuned_models[best_model_name].get_params())
best_model_obj.fit(X, y)
explainer = shap.TreeExplainer(best_model_obj)
shap_vals = explainer.shap_values(X)

fig, ax = plt.subplots(figsize=(7, 4))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='dot', show=False, color_bar=True)
plt.title(f'Vanillylamine — SHAP Feature Importance ({best_model_name}, Tuned)',
          fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_SHAP_beeswarm_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_SHAP_beeswarm_tuned.png")

fig, ax = plt.subplots(figsize=(6, 3.5))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='bar', show=False)
plt.title(f'Vanillylamine — Mean |SHAP| ({best_model_name})', fontsize=12)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_SHAP_bar_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_SHAP_bar_tuned.png")

# Figure 6 — Feature scatter plots (raw scale)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, feature_names):
    ax.scatter(data_raw[feat].values, data_raw['nanogramPerG'].values,
               color='#2ca02c', alpha=0.7, s=60,
               edgecolors='white', lw=0.5)
    ax.set_xlabel(f'{feat} concentration (mg/L)', fontsize=11)
    ax.set_ylabel('Vanillylamine yield (ng/g)', fontsize=11)
    ax.set_title(f'Vanillylamine vs {feat}', fontsize=12)
    ax.grid(True, alpha=0.3); ax.tick_params(labelsize=10)
plt.suptitle('Vanillylamine — Feature Scatter Plots', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Vanillylamine_FeaturePlots.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Vanillylamine_FeaturePlots.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*65)
print("BEST HYPERPARAMETERS  (report in paper Section IV.D)")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

print("\n" + "="*65)
print("SUMMARY: LOO CV METRICS  (tuned) — copy into paper table")
print("="*65)
print(f"{'Model':<22} {'R²':>6} {'MAE':>7} {'RMSE':>8}")
print("-"*46)
for name in loo_results:
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>6.2f} {r['mae']:>7.2f} {r['rmse']:>8.2f}")

print(f"\nBest model: {best_model_name} (LOO R²={loo_results[best_model_name]['r2']:.2f})")
print("\nAll figures saved. Done.")

Dataset: 35 samples, 3 features
y range: -1.141 to 2.670  (standardised)
NOTE: n=15. LOO CV is the primary evaluation metric.

CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)

Random Forest: searching 36 combinations...
  10/36 done — best so far: 0.497
  20/36 done — best so far: 0.507
  30/36 done — best so far: 0.507
  36/36 done — best so far: 0.513
  → Best params: {'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 2, 'random_state': 42}
  → Best LOO R²: 0.513

Gradient Boosting: searching 27 combinations...
  10/27 done — best so far: 0.663
  20/27 done — best so far: 0.663
  27/27 done — best so far: 0.663
  → Best params: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'random_state': 42}
  → Best LOO R²: 0.663

XGBoost: searching 162 combinations...
  10/162 done — best so far: 0.546
  20/162 done — best so far: 0.609
  30/162 done — best so far: 0.609
  40/162 done — best so far: 0.609
  50/162 done — best so far: 0.609
  60/162 done — best so far

dihyrdrocapsaicin

In [9]:
# =============================================================================
# DIHYDROCAPSAICIN - IMPROVED ANALYSIS WITH CROSS-VALIDATED HYPERPARAMETER TUNING
# Input files:
#   - DiCap_standardized_data.xlsx   (35-row standardised dataset, Sheet1)
#   - DiCap_changed.xlsx             (raw data for feature scatter plots, Sheet3)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import itertools

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import shap

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'Random Forest':      '#1f77b4',
    'Gradient Boosting':  '#2ca02c',
    'XGBoost':            '#ff7f0e',
    'CatBoost':           '#d62728',
}
MARKERS = {'9:1': 'o', '8:2': '^', '7:3': 's'}

# =============================================================================
# 1. LOAD DATA
# =============================================================================
data     = pd.read_excel('DiCap_standardized_data.xlsx')
data_raw = pd.read_excel('DiCap_changed.xlsx', sheet_name=2)  # Sheet3 (0-indexed)

data = data[['TIBA', 'TDZ', 'BA', 'nanogramPerG']].dropna()

X = data[['TIBA', 'TDZ', 'BA']].values
y = data['nanogramPerG'].values
feature_names = ['TIBA', 'TDZ', 'BA']

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"y range: {y.min():.3f} to {y.max():.3f}  (standardised)")

# =============================================================================
# 2. MANUAL LOO CV GRID SEARCH
# =============================================================================

def loo_cv_score(model_class, params, X, y):
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y)
    for train_idx, test_idx in loo.split(X):
        m = model_class(**params)
        m.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = m.predict(X[test_idx])
    return r2_score(y, y_pred), y_pred


def grid_search_loo(model_class, param_grid, X, y, label):
    keys   = list(param_grid.keys())
    values = list(param_grid.values())
    combos = list(itertools.product(*values))

    best_score  = -np.inf
    best_params = None
    total = len(combos)

    print(f"\n{label}: searching {total} combinations...")
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        try:
            score, _ = loo_cv_score(model_class, params, X, y)
            if score > best_score:
                best_score  = score
                best_params = params
        except Exception:
            pass
        if i % 10 == 0 or i == total:
            print(f"  {i}/{total} done — best so far: {best_score:.3f}", flush=True)

    print(f"  → Best params: {best_params}")
    print(f"  → Best LOO R²: {best_score:.3f}")
    return best_params, best_score


print("\n" + "="*65)
print("CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)")
print("="*65)

rf_params, rf_score = grid_search_loo(
    RandomForestRegressor,
    {'n_estimators': [50, 100, 200], 'max_depth': [None, 3, 5, 7],
     'min_samples_split': [2, 4, 6], 'random_state': [42]},
    X, y, 'Random Forest'
)

gb_params, gb_score = grid_search_loo(
    GradientBoostingRegressor,
    {'n_estimators': [100, 200, 300], 'learning_rate': [0.05, 0.1, 0.2],
     'max_depth': [2, 3, 4], 'random_state': [42]},
    X, y, 'Gradient Boosting'
)

xgb_params, xgb_score = grid_search_loo(
    XGBRegressor,
    {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2],
     'max_depth': [3, 4, 6], 'min_child_weight': [1, 3, 5],
     'subsample': [0.8, 1.0], 'random_state': [42], 'verbosity': [0]},
    X, y, 'XGBoost'
)

cb_params, cb_score = grid_search_loo(
    CatBoostRegressor,
    {'iterations': [500, 1000], 'learning_rate': [0.01, 0.05, 0.1],
     'depth': [4, 6, 8], 'loss_function': ['RMSE'],
     'random_state': [42], 'verbose': [0]},
    X, y, 'CatBoost'
)

print("\n" + "="*65)
print("TUNING COMPLETE")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

# =============================================================================
# 3. BUILD TUNED MODELS
# =============================================================================
tuned_models = {
    'Random Forest':     RandomForestRegressor(**rf_params),
    'Gradient Boosting': GradientBoostingRegressor(**gb_params),
    'XGBoost':           XGBRegressor(**xgb_params),
    'CatBoost':          CatBoostRegressor(**cb_params),
}

splits = {'9:1': 0.1, '8:2': 0.2, '7:3': 0.3}

# =============================================================================
# 4. TRAIN-TEST SPLIT RESULTS
# =============================================================================
print("\n" + "="*65)
print("TRAIN-TEST SPLIT METRICS  (tuned models)")
print("="*65)

split_results = {}

for split_label, test_size in splits.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42)
    split_results[split_label] = {}

    print(f"\n── Split {split_label} ──")
    print(f"{'Model':<22} {'Train R²':>9} {'Train MAE':>10} {'Train RMSE':>11}"
          f" {'Test R²':>8} {'Test MAE':>9} {'Test RMSE':>10}")
    print("-"*82)

    for name, model in tuned_models.items():
        m = model.__class__(**model.get_params())
        m.fit(X_tr, y_tr)
        ytr_p = m.predict(X_tr)
        yte_p = m.predict(X_te)

        res = {
            'train_r2':   r2_score(y_tr, ytr_p),
            'train_mae':  mean_absolute_error(y_tr, ytr_p),
            'train_rmse': np.sqrt(mean_squared_error(y_tr, ytr_p)),
            'test_r2':    r2_score(y_te, yte_p),
            'test_mae':   mean_absolute_error(y_te, yte_p),
            'test_rmse':  np.sqrt(mean_squared_error(y_te, yte_p)),
            'y_test': y_te, 'y_pred': yte_p,
        }
        split_results[split_label][name] = res
        print(f"{name:<22} {res['train_r2']:>9.2f} {res['train_mae']:>10.2f} "
              f"{res['train_rmse']:>11.2f} {res['test_r2']:>8.2f} "
              f"{res['test_mae']:>9.2f} {res['test_rmse']:>10.2f}")

# =============================================================================
# 5. FINAL LOO CV
# =============================================================================
print("\n" + "="*65)
print("LEAVE-ONE-OUT CV RESULTS  (tuned models, Dihydrocapsaicin)")
print("="*65)
print(f"{'Model':<22} {'LOO R²':>8} {'LOO MAE':>9} {'LOO RMSE':>10}")
print("-"*52)

loo_results = {}
loo = LeaveOneOut()

for name, model in tuned_models.items():
    m = model.__class__(**model.get_params())
    y_loo_pred = cross_val_predict(m, X, y, cv=loo)
    loo_results[name] = {
        'r2':     r2_score(y, y_loo_pred),
        'mae':    mean_absolute_error(y, y_loo_pred),
        'rmse':   np.sqrt(mean_squared_error(y, y_loo_pred)),
        'y_pred': y_loo_pred,
    }
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>8.2f} {r['mae']:>9.2f} {r['rmse']:>10.2f}")

# =============================================================================
# FIGURES
# =============================================================================

# Figure 1 — Actual vs Predicted (all splits)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, split_label in zip(axes, splits):
    for name, color in COLORS.items():
        res = split_results[split_label][name]
        ax.scatter(res['y_test'], res['y_pred'], color=color,
                   alpha=0.75, s=45, marker=MARKERS[split_label],
                   label=name, zorder=3)
    lo, hi = y.min()-0.1, y.max()+0.1
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,label='Perfect fit',zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual values (standardised)')
    ax.set_title(f'Split {split_label}')
    ax.grid(True,alpha=0.3); ax.set_aspect('equal',adjustable='box')
axes[0].set_ylabel('Predicted values (standardised)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           frameon=True, bbox_to_anchor=(0.5,-0.08))
fig.suptitle('Dihydrocapsaicin — Actual vs. Predicted (Tuned)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_ActualVsPredicted_tuned.png', bbox_inches='tight')
plt.close()
print("\nSaved: Fig_Dihydrocapsaicin_ActualVsPredicted_tuned.png")

# Figure 2 — LOO CV actual vs predicted
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    ax.scatter(y, res['y_pred'], color=color, alpha=0.7, s=40, zorder=3)
    lo, hi = y.min()-0.1, y.max()+0.1
    ax.plot([lo,hi],[lo,hi],'k--',lw=1.2,zorder=2)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual (standardised)')
    ax.set_title(f'{name}\nLOO R²={res["r2"]:.2f}', fontsize=10)
    ax.grid(True,alpha=0.3); ax.set_aspect('equal',adjustable='box')
axes[0].set_ylabel('Predicted (standardised)')
fig.suptitle('Dihydrocapsaicin — Leave-One-Out CV (Tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_LOO_CV_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_LOO_CV_tuned.png")

# Figure 3 — Residuals
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (name, color) in zip(axes, COLORS.items()):
    res = loo_results[name]
    residuals = y - res['y_pred']
    ax.scatter(res['y_pred'], residuals, color=color, alpha=0.7, s=40, zorder=3)
    ax.axhline(0, color='black', linestyle='--', lw=1.2, zorder=2)
    ax.set_xlabel('Predicted (standardised)')
    ax.set_title(name, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Residual (actual − predicted)')
fig.suptitle('Dihydrocapsaicin — Residual Analysis (LOO CV, Tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_Residuals_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_Residuals_tuned.png")

# Figure 4 — LOO R² bar chart
fig, ax = plt.subplots(figsize=(7, 4))
names   = list(loo_results.keys())
r2_vals = [loo_results[n]['r2'] for n in names]
colors  = [COLORS[n] for n in names]
bars = ax.bar(names, r2_vals, color=colors, width=0.5,
              edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, r2_vals):
    ypos = max(bar.get_height(), 0) + 0.01
    ax.text(bar.get_x()+bar.get_width()/2, ypos,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(min(0, min(r2_vals))-0.1, 1.1)
ax.set_ylabel('LOO Cross-Validation R²')
ax.set_title('Dihydrocapsaicin — Model Comparison (LOO CV, Tuned)')
ax.axhline(0.7, color='grey', linestyle=':', lw=1, label='R²=0.70 reference')
ax.axhline(0, color='black', linestyle='-', lw=0.8)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_LOO_R2_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_LOO_R2_tuned.png")

# Figure 5 — SHAP (best model by LOO R²)
best_model_name = max(loo_results, key=lambda n: loo_results[n]['r2'])
best_model      = tuned_models[best_model_name]
print(f"\nComputing SHAP values ({best_model_name}, tuned)...")

bm = best_model.__class__(**best_model.get_params())
bm.fit(X, y)
explainer = shap.TreeExplainer(bm)
shap_vals = explainer.shap_values(X)

fig, ax = plt.subplots(figsize=(7, 4))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='dot', show=False, color_bar=True)
plt.title(f'Dihydrocapsaicin — SHAP Feature Importance ({best_model_name}, Tuned)',
          fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_SHAP_beeswarm_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_SHAP_beeswarm_tuned.png")

fig, ax = plt.subplots(figsize=(6, 3.5))
shap.summary_plot(shap_vals, X, feature_names=feature_names,
                  plot_type='bar', show=False)
plt.title(f'Dihydrocapsaicin — Mean |SHAP| ({best_model_name})', fontsize=12)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_SHAP_bar_tuned.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_SHAP_bar_tuned.png")

# Figure 6 — Feature scatter plots (raw scale)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, feature_names):
    ax.scatter(data_raw[feat].values, data_raw['nanogramPerG'].values,
               color='#d62728', alpha=0.7, s=50, edgecolors='white', lw=0.5)
    ax.set_xlabel(f'{feat} concentration (mg/L)', fontsize=11)
    ax.set_ylabel('Dihydrocapsaicin yield (ng/g)', fontsize=11)
    ax.set_title(f'Dihydrocapsaicin vs {feat}', fontsize=12)
    ax.grid(True, alpha=0.3); ax.tick_params(labelsize=10)
plt.suptitle('Dihydrocapsaicin — Feature Scatter Plots', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Fig_Dihydrocapsaicin_FeaturePlots.png', bbox_inches='tight')
plt.close()
print("Saved: Fig_Dihydrocapsaicin_FeaturePlots.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*65)
print("BEST HYPERPARAMETERS  (report in paper Section IV.D)")
print("="*65)
print(f"Random Forest:     {rf_params}  (LOO R²={rf_score:.3f})")
print(f"Gradient Boosting: {gb_params}  (LOO R²={gb_score:.3f})")
print(f"XGBoost:           {xgb_params}  (LOO R²={xgb_score:.3f})")
print(f"CatBoost:          {cb_params}  (LOO R²={cb_score:.3f})")

print("\n" + "="*65)
print("SUMMARY: LOO CV METRICS  (tuned) — copy into paper table")
print("="*65)
print(f"{'Model':<22} {'R²':>6} {'MAE':>7} {'RMSE':>8}")
print("-"*46)
for name in loo_results:
    r = loo_results[name]
    print(f"{name:<22} {r['r2']:>6.2f} {r['mae']:>7.2f} {r['rmse']:>8.2f}")

print(f"\nBest model by LOO R²: {best_model_name}")
print("\nAll figures saved. Done.")

Dataset: 35 samples, 3 features
y range: -1.626 to 2.068  (standardised)

CROSS-VALIDATED HYPERPARAMETER TUNING  (manual LOO loop)

Random Forest: searching 36 combinations...
  10/36 done — best so far: 0.446
  20/36 done — best so far: 0.446
  30/36 done — best so far: 0.446
  36/36 done — best so far: 0.446
  → Best params: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'random_state': 42}
  → Best LOO R²: 0.446

Gradient Boosting: searching 27 combinations...
  10/27 done — best so far: 0.471
  20/27 done — best so far: 0.471
  27/27 done — best so far: 0.471
  → Best params: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'random_state': 42}
  → Best LOO R²: 0.471

XGBoost: searching 162 combinations...
  10/162 done — best so far: 0.456
  20/162 done — best so far: 0.458
  30/162 done — best so far: 0.458
  40/162 done — best so far: 0.458
  50/162 done — best so far: 0.458
  60/162 done — best so far: 0.458
  70/162 done — best so far: 0.458
  80/1